# 机器学习基线对照实验（ML_Funning）

## 目的
使用逻辑回归模型作为机器学习基线进行文本分类实验，作为BERT模型的对照基线。使用与LoRA实验一致的数据划分与超参数，输出统一指标格式。

## 数据流向
输入：原始数据文件（waimai.csv）-> TF-IDF特征提取 -> 逻辑回归模型训练 -> 模型评估
输出：训练好的逻辑回归模型、训练过程指标、测试集评估结果

## 操作步骤
1. 导入必要的库和配置
2. 使用TF-IDF向量化器提取文本特征
3. 训练逻辑回归模型
4. 在训练集、验证集、测试集上评估模型性能

In [1]:
"""
目的：导入逻辑回归实验所需的所有Python库和模块

数据流向：
输入：无（直接导入库）
输出：所有必要的库和函数已加载到当前命名空间

操作步骤：
1. 导入系统库：time（时间计算）
2. 导入sklearn特征提取：TfidfVectorizer（TF-IDF向量化器）
3. 导入sklearn模型：LogisticRegression（逻辑回归分类器）
4. 导入sklearn评估指标：accuracy_score（准确率）、precision_recall_fscore_support（精确率、召回率、F1）、log_loss（对数损失）
5. 导入自定义配置：从Bert_Config导入统一配置和工具函数（数据加载、分割）
"""

import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data

print("✅ 所有库导入完成")

C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [2]:
"""
目的：从统一配置文件加载实验参数，设置逻辑回归模型的特定配置参数

数据流向：
输入：CONFIG字典（从Bert_Config导入）-> 提取配置参数
输出：所有配置变量已设置，配置信息已打印

操作步骤：
1. 从CONFIG字典提取通用配置：数据路径、随机种子
2. 从CONFIG字典提取TF-IDF相关参数：
   - MAX_FEATURES：最大特征数（控制特征维度）
   - NGRAM_RANGE：N-gram范围（字符级n-gram特征）
   - MIN_DF：最小文档频率（过滤低频特征）
   - MAX_DF：最大文档频率（过滤高频特征，如停用词）
3. 设置模型名称：MODEL_NAME = "LogisticRegression-ML"
4. 设置实验名称：EXP_NAME = "机器学习基线对照实验"
5. 打印所有配置参数，便于确认实验设置
"""

MODEL_NAME = "LogisticRegression-ML"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]

MAX_FEATURES = CONFIG["CLASSIC_MAX_FEATURES"]
NGRAM_RANGE = CONFIG["CLASSIC_NGRAM_RANGE"]
MIN_DF = CONFIG["CLASSIC_MIN_DF"]
MAX_DF = CONFIG["CLASSIC_MAX_DF"]

EXP_NAME = "机器学习基线对照实验"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"模型名称: {MODEL_NAME}")
print(f"数据目录: {DATA_PATH}")
print(f"随机种子: {RANDOM_SEED}")
print(f"最大特征数: {MAX_FEATURES}")
print(f"N-gram范围: {NGRAM_RANGE}")
print(f"最小文档频率: {MIN_DF}")
print(f"最大文档频率: {MAX_DF}")
print("=" * 50)

✅ 配置加载完成


In [3]:
"""
目的：定义工具函数，包括TF-IDF向量化器构建、评估函数和报告打印函数

数据流向：
输入：配置参数 -> 创建函数对象
输出：函数对象（可以在后续代码中调用）

操作步骤：
1. 定义build_vectorizer函数：构建TF-IDF向量化器
2. 定义evaluate函数：计算多项评估指标
3. 定义print_report函数：格式化打印评估结果
"""

def build_vectorizer():
    """
    目的：构建TF-IDF向量化器，用于将文本转换为数值特征向量
    
    输入：无（使用全局配置参数MAX_FEATURES、NGRAM_RANGE、MIN_DF、MAX_DF）
    输出：TfidfVectorizer对象（已配置好的向量化器）
    
    数据流向：
    配置参数 -> 创建TfidfVectorizer对象 -> 返回向量化器
    
    操作步骤：
    1. 创建TfidfVectorizer对象，设置以下参数：
       - analyzer="char": 按字符级别分析（适合中文文本）
       - ngram_range: N-gram范围，例如(1,3)表示提取1-gram、2-gram、3-gram特征
       - max_features: 保留的最大特征数量，控制特征维度
       - min_df: 最小文档频率，过滤掉出现次数太少的特征
       - max_df: 最大文档频率，过滤掉出现次数太多的特征（如停用词）
    2. 返回配置好的向量化器对象
    """
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
        min_df=MIN_DF,
        max_df=MAX_DF,
    )


def evaluate(y_true, y_pred, y_prob=None):
    """
    目的：计算模型在数据集上的多项评估指标
    
    输入：
      - y_true: 真实标签（数组或列表）
      - y_pred: 预测标签（数组或列表）
      - y_prob: 预测概率（可选，用于计算损失）
    输出：
      - (acc, precision, recall, f1, loss) 五元组
    
    数据流向：
    输入真实标签、预测标签、预测概率 -> 计算各项指标 -> 返回指标元组
    
    操作步骤：
    1. 使用sklearn计算准确率（accuracy）：正确预测的样本比例
    2. 使用sklearn计算精确率、召回率、F1分数：
       - precision: 预测为正的样本中真正为正的比例
       - recall: 真正为正的样本中被正确预测的比例
       - f1: 精确率和召回率的调和平均数
       - average="binary": 针对二分类任务
       - zero_division=0: 避免除零错误
    3. 如果提供了预测概率，使用log_loss计算交叉熵损失
    4. 返回所有指标的元组
    """
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    loss = None
    if y_prob is not None:
        loss = log_loss(y_true, y_prob)
    return acc, precision, recall, f1, loss


def print_report(train_metrics, val_metrics, test_metrics, elapsed_sec):
    """
    目的：格式化打印训练、验证、测试集的评估结果
    
    输入：
      - train_metrics: 训练集指标元组 (acc, precision, recall, f1, loss)
      - val_metrics: 验证集指标元组
      - test_metrics: 测试集指标元组
      - elapsed_sec: 训练耗时（秒）
    输出：无（直接打印到控制台）
    
    数据流向：
    输入指标元组和训练时间 -> 格式化输出 -> 打印到控制台
    
    操作步骤：
    1. 将训练时间从秒转换为分钟
    2. 定义辅助函数format_loss：格式化loss值，处理None情况
    3. 打印分隔线和模型名称
    4. 依次打印训练集、验证集、测试集的五项指标：
       - Acc: 准确率
       - Precision: 精确率
       - Recall: 召回率
       - F1: F1分数
       - Loss: 交叉熵损失（如果为None则显示N/A）
    5. 打印训练时间（秒和分钟）
    6. 打印结束分隔线
    """
    elapsed_min = elapsed_sec / 60
    
    # 辅助函数：格式化loss值，处理None情况
    def format_loss(loss):
        return f"{loss:.3f}" if loss is not None else "N/A"
    
    print("\n" + "=" * 50)
    print(f"模型: {MODEL_NAME}")
    print(
        f"训练集 - Acc: {train_metrics[0]:.3f} | Precision: {train_metrics[1]:.3f} | "
        f"Recall: {train_metrics[2]:.3f} | F1: {train_metrics[3]:.3f} | "
        f"Loss: {format_loss(train_metrics[4])}"
    )
    print(
        f"验证集 - Acc: {val_metrics[0]:.3f} | Precision: {val_metrics[1]:.3f} | "
        f"Recall: {val_metrics[2]:.3f} | F1: {val_metrics[3]:.3f} | "
        f"Loss: {format_loss(val_metrics[4])}"
    )
    print(
        f"测试集 - Acc: {test_metrics[0]:.3f} | Precision: {test_metrics[1]:.3f} | "
        f"Recall: {test_metrics[2]:.3f} | F1: {test_metrics[3]:.3f} | "
        f"Loss: {format_loss(test_metrics[4])}"
    )
    print(f"训练时间: {elapsed_sec:.1f} 秒 ({elapsed_min:.2f} 分钟)")
    print("=" * 50)

print("✅ 工具函数定义完成")

✅ 工具函数定义完成


In [4]:
"""
目的：加载原始数据，使用TF-IDF向量化器提取特征，准备训练、验证和测试数据

数据流向：
输入：DATA_PATH（数据文件路径）、RANDOM_SEED（随机种子）
-> 加载数据 -> 划分数据集 -> TF-IDF特征提取 -> 提取标签
输出：x_train、x_val、x_test（特征矩阵）、y_train、y_val、y_test（标签数组）

操作步骤：
1. 设置随机种子，确保数据划分可复现
2. 调用load_raw_data加载原始CSV数据
3. 调用split_data将数据按8:1:1划分为训练集、验证集、测试集
4. 调用build_vectorizer创建TF-IDF向量化器
5. 使用fit_transform在训练集上拟合向量化器并转换训练集文本为特征矩阵
6. 使用transform转换验证集和测试集文本为特征矩阵（使用训练集拟合的向量化器）
7. 提取训练集、验证集、测试集的标签，转换为整数数组
8. 打印数据准备完成提示
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vectorizer = build_vectorizer()
x_train = vectorizer.fit_transform(train_df["review"])
x_val = vectorizer.transform(val_df["review"])
x_test = vectorizer.transform(test_df["review"])

y_train = train_df["label"].astype(int).values
y_val = val_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

print("✅ 数据准备完成")



✅ 数据准备完成


In [ ]:
"""
目的：创建逻辑回归模型，训练模型，并在训练集、验证集、测试集上评估模型性能

数据流向：
输入：x_train、y_train（训练特征和标签）、x_val、y_val（验证特征和标签）、x_test、y_test（测试特征和标签）
-> 创建模型 -> 训练模型 -> 预测 -> 评估指标
输出：训练好的模型、训练时间、各项评估指标

操作步骤：
1. 创建逻辑回归模型：LogisticRegression，设置最大迭代次数和随机种子
2. 记录训练开始时间
3. 使用训练集特征和标签训练模型：model.fit(x_train, y_train)
4. 计算训练耗时：elapsed = 当前时间 - 开始时间
5. 在训练集、验证集、测试集上进行预测：使用predict方法得到预测标签
6. 在训练集、验证集、测试集上获取预测概率：使用predict_proba方法得到类别概率
7. 调用evaluate函数计算训练集、验证集、测试集的评估指标
8. 打印实验名称
9. 调用print_report函数格式化打印所有评估结果
"""

model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)

start = time.time()
model.fit(x_train, y_train)
elapsed = time.time() - start

train_pred = model.predict(x_train)
val_pred = model.predict(x_val)
test_pred = model.predict(x_test)

train_prob = model.predict_proba(x_train)
val_prob = model.predict_proba(x_val)
test_prob = model.predict_proba(x_test)

train_metrics = evaluate(y_train, train_pred, train_prob)
val_metrics = evaluate(y_val, val_pred, val_prob)
test_metrics = evaluate(y_test, test_pred, test_prob)
print("\n实验名称:"+EXP_NAME)
print_report(train_metrics, val_metrics, test_metrics, elapsed)
